In [ ]:

!pip -q install transformer-lens


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformer_lens.model_bridge import TransformerBridge


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("PyTorch:", torch.__version__)


Device: cpu
PyTorch: 2.14.1+cpu


In [ ]:
# Load GPT-2 Medium
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerBridge.boot_transformers(
    "gpt2-medium",
    device=device
)

model.eval()


print("Layers:", model.cfg.n_layers)
print("Heads per layer:", model.cfg.n_heads)
print("Residual Dimension:", model.cfg.d_model)
print("Head dimension:", model.cfg.d_head)

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.52GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Layers: 24
Heads per layer: 16
Residual Dimension: 1024
Head dimension: 64


In [ ]:
model.to_tokens("Patricia")

tensor([[50256,   380]])

In [ ]:
name_pairs = [
    ("Alice", "Bob"),
    ("Mary", "John"),
    ("David", "Michael"),
    ("Sarah", "James"),
    ("Robert", "William"),
    ("Emma", "Daniel"),
    ("Linda", "Richard"),
    ("Jennifer", "Thomas"),
    ("Patricia", "Charles"),
    ("Barbara", "Christopher")
]

prompt = [
    f"When {a} and {b} went to the store, "
    f"{b} gave a drink to"
    for a, b in name_pairs
]


correct_names = [a for a, b in name_pairs]
incorrect_names = [b for a, b in name_pairs]

dataset = pd.DataFrame({
    "prompt": prompt,
    "correct_name": correct_names,
    "incorrect_name": incorrect_names
})

dataset.head()

,prompt,correct_name,incorrect_name
0,"When Alice and Bob went to the store, Bob gave...",Alice,Bob
1,"When Mary and John went to the store, John gav...",Mary,John
2,"When David and Michael went to the store, Mich...",David,Michael
3,"When Sarah and James went to the store, James ...",Sarah,James
4,"When Robert and William went to the store, Wil...",Robert,William


In [ ]:
# Here the tokenization needs to be verified


tokens = model.to_tokens(prompt)

print("Token shape:", tokens.shape)

for prompt, token_row in zip(dataset["prompt"], tokens):
    print("\n", prompt)
    print(model.to_str_tokens(token_row))


assert len(set(tokens.shape[1:] for _ in prompt)) == 1, "All prompts must have the same length"

Token shape: torch.Size([10, 15])

 When Alice and Bob went to the store, Bob gave a drink to
['<|endoftext|>', 'When', ' Alice', ' and', ' Bob', ' went', ' to', ' the', ' store', ',', ' Bob', ' gave', ' a', ' drink', ' to']

 When Mary and John went to the store, John gave a drink to
['<|endoftext|>', 'When', ' Mary', ' and', ' John', ' went', ' to', ' the', ' store', ',', ' John', ' gave', ' a', ' drink', ' to']

 When David and Michael went to the store, Michael gave a drink to
['<|endoftext|>', 'When', ' David', ' and', ' Michael', ' went', ' to', ' the', ' store', ',', ' Michael', ' gave', ' a', ' drink', ' to']

 When Sarah and James went to the store, James gave a drink to
['<|endoftext|>', 'When', ' Sarah', ' and', ' James', ' went', ' to', ' the', ' store', ',', ' James', ' gave', ' a', ' drink', ' to']

 When Robert and William went to the store, William gave a drink to
['<|endoftext|>', 'When', ' Robert', ' and', ' William', ' went', ' to', ' the', ' store', ',', ' William',

In [ ]:
# verify explicitly target name should be one token with its leading space

for name in set(correct_names + incorrect_names):
    token_id = model.to_single_token(" " + name)
    print(name, "->", token_id)

William -> 3977
Bob -> 5811
Thomas -> 5658
Jennifer -> 16348
Sarah -> 10490
James -> 3700
Alice -> 14862
John -> 1757
Patricia -> 30223
David -> 3271
Christopher -> 12803
Richard -> 6219
Linda -> 21776
Barbara -> 16685
Michael -> 3899
Robert -> 5199
Emma -> 18966
Charles -> 7516
Daniel -> 7806
Mary -> 5335


In [ ]:
# Calculate logit IOT difference per-example

# Make sure the baseline will perfect, if otherwise improve the prompt distribution before proceeding.

def get_ioi_scores(logits, correct_names, incorrect_names):
    """
    Calculate per-example IOI logit differences.
    """

    scores = []

    for i, (correct, incorrect) in enumerate(
        zip(correct_names, incorrect_names)
    ):

        correct_id = model.to_single_token(" " + correct)
        incorrect_id = model.to_single_token(" " + incorrect)

        correct_logit = logits[i, -1, correct_id]
        incorrect_logit = logits[i, -1, incorrect_id]

        # here we will use logit metric
        scores.append(
            (correct_logit - incorrect_logit).item()
        )

    return np.array(scores)


def evaluate_ioi(logits):
    scores = get_ioi_scores(
        logits,
        correct_names,
        incorrect_names
    )

    return {
        "mean_logit_difference": scores.mean(),
        "accuracy": (scores > 0).mean(),
        "individual_scores": scores
    }


clean_logits = model(tokens)

baseline = evaluate_ioi(clean_logits)


print("Baseline logit difference:",
      baseline["mean_logit_difference"])

print("Baseline accuracy:",
      baseline["accuracy"])

Baseline logit difference: 3.9765132904052733
Baseline accuracy: 1.0


In [ ]:
# Cache Attention-Head Activations

# [batch, position, head, d_head]

# We have another edge here, individual attention head could be inspected
# rather than replacing entire attention layer.


# now you can access all the outputs of attention heads

# this is reference cache containing the original computation
# a clean experimental condition
clean_logits, clean_cache = model.run_with_cache(
    tokens,
    names_filter=lambda name: name.endswith("attn.hook_z")
)

for layer in range(model.cfg.n_layers):
    name = f"blocks.{layer}.attn.hook_z"
    activation = clean_cache[name]

    print(
        f"Layer {layer}:",
        tuple(activation.shape)
    )




Layer 0: (10, 15, 16, 64)
Layer 1: (10, 15, 16, 64)
Layer 2: (10, 15, 16, 64)
Layer 3: (10, 15, 16, 64)
Layer 4: (10, 15, 16, 64)
Layer 5: (10, 15, 16, 64)
Layer 6: (10, 15, 16, 64)
Layer 7: (10, 15, 16, 64)
Layer 8: (10, 15, 16, 64)
Layer 9: (10, 15, 16, 64)
Layer 10: (10, 15, 16, 64)
Layer 11: (10, 15, 16, 64)
Layer 12: (10, 15, 16, 64)
Layer 13: (10, 15, 16, 64)
Layer 14: (10, 15, 16, 64)
Layer 15: (10, 15, 16, 64)
Layer 16: (10, 15, 16, 64)
Layer 17: (10, 15, 16, 64)
Layer 18: (10, 15, 16, 64)
Layer 19: (10, 15, 16, 64)
Layer 20: (10, 15, 16, 64)
Layer 21: (10, 15, 16, 64)
Layer 22: (10, 15, 16, 64)
Layer 23: (10, 15, 16, 64)


In [ ]:
def ablate_head(z, hook, head_index):
    """
    Ablate one attention head's output at every position.
    z shape: [batch, position, head, d_head]
    """
    z = z.clone()
    z[:, :, head_index, :] = 0

    return z


def run_head_ablation(
    tokens,
    layer,
    head
):
    hook_name = f"blocks.{layer}.attn.hook_z"

    logits = model.run_with_hooks(
        tokens,
        fwd_hooks=[
            (
                hook_name,
                lambda z, hook: ablate_head(
                    z, hook, head
                )
            )
        ]
    )

    return evaluate_ioi(logits)


In [ ]:
# testing one head straightaway to see the effect of ablating it.
# please consider as intervention, to perform a behavior analysis.


test_layer = 9
test_head = 9

result = run_head_ablation(
    tokens,
    test_layer,
    test_head
)

print("Original score:",
      baseline["mean_logit_difference"])

print("Ablated score:",
      result["mean_logit_difference"])

print("Score change:",
      result["mean_logit_difference"]
      - baseline["mean_logit_difference"])


Original score: 3.9765132904052733
Ablated score: 4.015217590332031
Score change: 0.038704299926757724


In [ ]:
# now let's play a big ablating
# ablating and sweeping all the heads of each layer

# what we are measuring is

# Layer index.

# Head index.

# Original logit difference.

# Ablated logit difference.

# Difference between the two.

# Accuracy after ablation.



In [ ]:
results = []

baseline_score = baseline["mean_logit_difference"]

for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):

        result = run_head_ablation(
            tokens,
            layer,
            head
        )

        ablated_score = result["mean_logit_difference"]

        results.append({
            "layer": layer,
            "head": head,
            "baseline_score": baseline_score,
            "ablated_score": ablated_score,
            "score_change": ablated_score - baseline_score,
            "accuracy": result["accuracy"]
        })

results_df = pd.DataFrame(results)

results_df.to_csv(
    "gpt2_head_ablation_results.csv",
    index=False
)

display(
    results_df.sort_values(
        "score_change"
    ).head(20)
)


,layer,head,baseline_score,ablated_score,score_change,accuracy
305,19,1,3.976513,3.361216,-0.615298,1.0
244,15,4,3.976513,3.416105,-0.560408,1.0
248,15,8,3.976513,3.536801,-0.439713,1.0
212,13,4,3.976513,3.628910,-0.347603,1.0
343,21,7,3.976513,3.687197,-0.289316,1.0
172,10,12,3.976513,3.715401,-0.261112,1.0
221,13,13,3.976513,3.734173,-0.242340,1.0
29,1,13,3.976513,3.754457,-0.222056,1.0
32,2,0,3.976513,3.756130,-0.220383,1.0
174,10,14,3.976513,3.771069,-0.205445,1.0
